# **Improvements made over V4:**
- Preprocessing input text to correct grammatical errors and unnecessary characters.
- Improving the prompt to shift the model's behaviour from being retrieval oriented to being generator oriented.

In [1]:
# Parameter values.
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
RETRIEVAL_STRATEGY = "mmr"
K = 5
FETCH_K = 10  # The number of chunks to be fetched before performing re-ranking.
MAX_NEW_TOKENS = 300
MIN_NEW_TOKENS = 100
REPETITION_PENALTY = 1.2
TEMPERATURE = 0.7
NO_REPEAT_N_GRAM_SIZE = 3

In [2]:
# Helper funciton.
import re

def clean_text(text):
    # Normalize whitespace
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")

    # Fix hyphenated line breaks (common in PDFs)
    text = re.sub(r'-\s+', '', text)

    # Remove weird special characters (keep useful punctuation)
    text = re.sub(r'[^a-zA-Z0-9.,;:()\- ]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# **Loading:**

In [3]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

# 🔹 Define data directory
DATA_DIR = Path("../data/raw")

# 🔹 Text cleaning function
def clean_text(text):
    # Fix malformed scientific notation
    text = text.replace("0.00-10", "0.00e-10")
    
    # Optional: add more cleaning rules here if needed
    return text

# 🔹 Load documents
documents = []

for file in DATA_DIR.rglob("*.pdf"):  # recursive search
    try:
        print(f"Loading: {file}")
        
        loader = PyMuPDFLoader(str(file))
        docs = loader.load()
        
        for doc in docs:
            # 🔹 Clean text
            doc.page_content = clean_text(doc.page_content)
            
            # 🔹 Add metadata
            doc.metadata["source_file"] = file.name
            doc.metadata["category"] = file.parent.name
        
        documents.extend(docs)

    except Exception as e:
        print(f"Skipping {file} due to error: {e}")

print(f"\nTotal documents loaded: {len(documents)}")

Loading: ..\data\raw\ML\Bias Variance tradeoff.pdf
Loading: ..\data\raw\ML\Bias-Variance Tradeoff_derivation_ons.pdf
Loading: ..\data\raw\ML\Chapter2-Regression-SimpleLinearRegressionAnalysis.pdf
Loading: ..\data\raw\ML\Clustering_ons - Clustering2_ons.pdf
Loading: ..\data\raw\ML\Clustering_ons - Clustering_ons-1.pdf
Loading: ..\data\raw\ML\Clustering_ons - Clustering_ons.pdf
Loading: ..\data\raw\ML\CNN_FML_ons-4.pdf
Loading: ..\data\raw\ML\Data_Science_Interview_Q_and_A_1690229703.pdf
Loading: ..\data\raw\ML\Decision Tree.pdf
Loading: ..\data\raw\ML\Dimesnsionality_reduction_ons-2.pdf
Loading: ..\data\raw\ML\Ensemble_Learning_ons.pdf
Loading: ..\data\raw\ML\Feed Forward Network ons-1.pdf
Loading: ..\data\raw\ML\Hands on Machine Learning.pdf
Loading: ..\data\raw\ML\Intro to probability(Logistic Regression).pdf
Loading: ..\data\raw\ML\K Nearest Neighbours.pdf
Loading: ..\data\raw\ML\Linear Algebra Basics.pdf
Loading: ..\data\raw\ML\MLE and MAP.pdf
Loading: ..\data\raw\ML\MLE vs MAP.pdf


# **Text Cleaning:**

In [4]:
for doc in documents:
    doc.page_content = clean_text(doc.page_content)

# **Chunking:**

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_documents(documents)

print(len(chunks))

5010


# **Embeddings:**

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# **Vector Store:**

In [7]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)

# **Retriever:**

In [8]:
retriever = vectorstore.as_retriever(
    search_type=RETRIEVAL_STRATEGY,
    search_kwargs={"k": K, "fetch_k": FETCH_K}
)

In [7]:
# Test retrieval.
query = "What is machine learning?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n--- Doc {i+1} ---\n")
    print(doc.page_content[:400])


--- Doc 1 ---

Here is a slightly more general definition:
[Machine Learning is the] field of study that gives computers the ability to learn
without being explicitly programmed.
—Arthur Samuel, 1959
And a more engineering-oriented one:
A computer program is said to learn from experience E with respect to some task T
and some performance measure P, if its performance on T, as measured by P, improves

--- Doc 2 ---

Why Use Machine Learning? 
| 
7

--- Doc 3 ---

main categories and fundamental concepts of Machine Learning systems?
• The main steps in a typical Machine Learning project.
• Learning by fitting a model to data.
• Optimizing a cost function.
• Handling, cleaning, and preparing data.
• Selecting and engineering features.
• Selecting a model and tuning hyperparameters using cross-validation.

--- Doc 4 ---

more). It was followed by hundreds of ML applications that now quietly power hun‐
dreds of products and features that you use regularly, from better recommendations
to vo

# **LLM:**

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype="float32"   # updated (torch_dtype deprecated)
)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


# **Prompt Building:**

In [10]:
def build_prompt(context, question):
    return f"""
You are a teacher explaining concepts clearly to a beginner.

You are given multiple sources of information.

INSTRUCTIONS:
- Read ALL sources carefully.
- Do NOT copy text directly.
- Combine ideas from different sources.
- Explain in simple and clear language.
- Break your answer into steps.
- Add intuition and examples.
- Ensure your answer is at least 5-6 sentences long.

CONTEXT:
{context}

QUESTION:
{question}

DETAILED ANSWER:
"""

# **Final Pipeline:**

In [11]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to("cpu")

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        min_new_tokens = MIN_NEW_TOKENS,
        do_sample=False,
        num_beams = 4,
        repetition_penalty=REPETITION_PENALTY,
        no_repeat_ngram_size=NO_REPEAT_N_GRAM_SIZE,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # DO NOT remove prompt for FLAN-T5
    return generated_text.strip()



def answer_query(query):
    # Retrieve documents
    docs = retriever.invoke(query)

    # Build context (with basic control)
    context = "\n\n".join([
        f"Source {i+1}:\n{doc.page_content[:500]}"   # trim to avoid overflow
        for i, doc in enumerate(docs)
    ])

    # Build prompt
    prompt = build_prompt(context, query)

    # Generate answer
    answer = generate_answer(prompt)

    return answer

# **Testing:**

In [13]:
print(answer_query("What is machine learning?"))

The field of study that gives computers the ability to learn without being explicitly programmed. —Arthur Samuel, 1959 And a more engineering-oriented one: A computer program is said to learn from experience E with respect to some task T and some performance measure P, if its performance on T, as measured by P, improves. Machine learning is the study of computer algorithms that improve automatically through experience. It is seen as a subset of artificial intelligence. Machine Learning explores the study and construction of algorithms that can learn from and make predictions on data.


#### **Observations:**
A lot of improvement but the system is still retrieval oriented rather than generation oriented and hence, response looks like random sentences stitched together.